In [ ]:
# [Setup]
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/code')

!pip install -q z3-solver evaluate python-Levenshtein bert_score "torchao>=0.16.0" joblib 2>/dev/null
import nltk
nltk.download('punkt_tab', quiet=True)

from src.selectors.pw_metrics import (
    pw_step1_z3_filter,
    pw_step2_n_unique,
    pw_step3_z3_le_count,
    pw_step4_mean_bleu,
    pw_step5_mean_bertscore,
    pw_step6_dedup_backtrans,
)

print('Ready.')

In [ ]:
# [CONFIG]
MODEL = "Qwen8b"    # "Qwen4b" | "Qwen8b" | "Ministral8b"
DATASET = "test"    # "test" | "test_folio" | "test_willow"
FORCE = False       # Set to True to recompute existing steps

print(f"Model:   {MODEL}")
print(f"Dataset: {DATASET}")
print(f"Force:   {FORCE}")

In [ ]:
# [Step 1/6] Z3 filter — keep only candidates whose FOL can be parsed by Z3
# Reads:  {tag}_metrics.json
# Writes: {tag}_pw_step1.json

pw_step1_z3_filter(model=MODEL, dataset=DATASET, force=FORCE)

In [ ]:
# [Step 2/6] n_unique — count unique FOLs among Z3-filtered candidates per sentence
# Reads:  {tag}_pw_step1.json
# Writes: {tag}_pw_step2.json

pw_step2_n_unique(model=MODEL, dataset=DATASET, force=FORCE)

In [ ]:
# [Step 3/6] z3_le_count — pairwise Z3 LE among filtered candidates (before dedup)
# Reads:  {tag}_pw_step2.json
# Writes: {tag}_pw_step3.json

pw_step3_z3_le_count(model=MODEL, dataset=DATASET, force=FORCE)

In [ ]:
# [Step 4/6] mean_bleu — mean pairwise FOL-token BLEU among FILTERED candidates only
# Reads:  {tag}_pw_step3.json
# Writes: {tag}_pw_step4.json

pw_step4_mean_bleu(model=MODEL, dataset=DATASET, force=FORCE)

In [ ]:
# [Step 5/6] mean_bertscore — mean pairwise BertScore F1 among FILTERED candidates only
# Reads:  {tag}_pw_step4.json
# Writes: {tag}_pw_step5.json

pw_step5_mean_bertscore(model=MODEL, dataset=DATASET, force=FORCE)

In [ ]:
# [Step 6/6] backtrans_sim — dedup per sentence, then back-translate + cosine sim
# Final output drops nl/gt_fol/fol, keeps only the 5 features + sentence_id + candidate_idx.
# Reads:  {tag}_pw_step5.json
# Writes: {tag}_pw_metrics.json (FINAL, consumed by M6 Stage 2)

pw_step6_dedup_backtrans(model=MODEL, dataset=DATASET, force=FORCE)

In [ ]:
# [Validation] Load final output and inspect
import json, os

# dataset → folder mapping (test=malls, test_folio=folio, test_willow=willow)
_ds = {"test": "malls", "test_folio": "folio", "test_willow": "willow"}.get(DATASET, DATASET)
_suffix = DATASET.replace("test_", "").replace("test", "")
_tag = f"{MODEL}_k10_{_suffix}" if _suffix else f"{MODEL}_k10"
out_path = f"/content/drive/MyDrive/code/data/results/{MODEL}/k10/{_ds}/{_tag}_pw_metrics.json"

if os.path.exists(out_path):
    with open(out_path) as f:
        data = json.load(f)
    print(f"Total rows: {len(data)}")
    print(f"Columns: {list(data[0].keys())}")
    print()
    print("Sample rows:")
    for row in data[:5]:
        print(f"  {row}")
else:
    print(f"Output not found: {out_path}")
    print("Run all 6 steps above first.")